# Lesson 25 Lab — Failure Modes: Outliers, Long Context, MoE, and Small Batches

**Puzzle:** Where should a quantized system be expected to fail first?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Failure modes map to mechanisms: range outliers, distribution shift, long-context cache/attention, MoE routing imbalance, and small irregular GEMMs.

### Core mechanism

One outlier can enlarge a group scale; shifted inputs change layer-output sensitivity; batch-one and routed experts reduce matrix sizes and make launch/dequant overhead visible.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "25-quantization-failure-modes"
device = require_cuda()
torch.manual_seed(2026 + 25)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Optimizing the average case can worsen a rare but critical slice. Global fallback is safe but expensive; targeted fallback needs reliable detection and routing.

### What this code tests

The lab holds weights fixed and stresses ordinary, outlier, shifted, and small-batch inputs, preserving each condition instead of averaging them together.

**Experiment:** Stress an INT4 linear reference with ordinary inputs, activation outliers, narrow batches, and shifted distributions on CUDA.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
w=torch.randn(1024,1024,device=device); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); rows=[]
cases={"ordinary":torch.randn(32,1024,device=device),"small_batch":torch.randn(1,1024,device=device),
       "activation_outliers":torch.randn(32,1024,device=device),"shifted_domain":torch.randn(32,1024,device=device)*3+2}
cases["activation_outliers"][:,::73]*=30
for name,x in cases.items(): rows.append({"case":name,"output_error":error_metrics(x@w.t(),x@dq.t()),
    "bf16_timing":cuda_benchmark(lambda:x@w.t(),warmup=3,repeats=12),"reference_w4_timing":cuda_benchmark(lambda:x@dq.t(),warmup=3,repeats=12)})
result=base_result(25,"pytorch-gpu"); result.update({"failure_matrix":rows,
    "conclusion":"Condition-specific tests exposed reversals that an aggregate average could conceal."})


## 3. Inspect the evidence

Keep a failure matrix by condition. Average error over mixed cases can conceal the exact reversal condition.

### Acceptance and rollback gate

Maintain a condition-by-metric failure matrix with reversal thresholds and reproduce each failure independently before assigning a fallback.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Condition-specific tests exposed reversals that an aggregate average could conceal.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:14+00:00",
  "failure_matrix": [
    {
      "bf16_timing": {
        "median_ms": 0.019584,
        "p90_ms": 0.020288,
        "repeats": 12,
        "samples_ms": [
          0.030048,
          0.022048,
          0.020288,
          0.01952,
          0.019744,
          0.019648,
          0.01888,
          0.019168,
          0.017984,
          0.019072,
          0.01968,
          0.019104
        ],
        "warmup": 3
      },
      "case": "ordinary",
      "output_error": {
        "cosine": 0.99317932,
        "mae": 2.97387171,
        "max_abs": 17.41661835,
        "rmse": 3.72911549
      },
 

## 4. Explain the result

Design negative tests from known mechanisms and preserve a fallback for the slice that fails.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).